# Execute SQL in BigQuery Using Service Account Credentials

This notebook:
1. Takes a service account JSON path and SQL file path as input
2. Creates a BigQuery client (`bq_client`) using service account credentials
3. Executes a DDL SQL script in BigQuery
4. Displays DDL job execution details

In [1]:
from pathlib import Path
from google.cloud import bigquery
from google.oauth2 import service_account

In [2]:
# Inputs
SERVICE_ACCOUNT_PATH = Path(r"C:\Users\p00102977\Desktop\Projects\API\repo\cognito-dev-380506-6ca25463100d.json")
SQL_FILE_PATH = Path(r"C:\Users\p00102977\Desktop\Projects\API\FInal Repo content\drl-cognito-ai\medical_potential\tolerability_and_patient_burden\ddl_scripts\clinical_trials_tolerability_data_ddl.sql")

if not SERVICE_ACCOUNT_PATH.exists():
    raise FileNotFoundError(f"Service account JSON not found: {SERVICE_ACCOUNT_PATH}")

if not SQL_FILE_PATH.exists():
    raise FileNotFoundError(f"SQL file not found: {SQL_FILE_PATH}")

print(f"Service account path: {SERVICE_ACCOUNT_PATH}")
print(f"SQL file path: {SQL_FILE_PATH}")

Service account path: C:\Users\p00102977\Desktop\Projects\API\repo\cognito-dev-380506-6ca25463100d.json
SQL file path: C:\Users\p00102977\Desktop\Projects\API\FInal Repo content\drl-cognito-ai\medical_potential\tolerability_and_patient_burden\ddl_scripts\clinical_trials_tolerability_data_ddl.sql


In [3]:
def create_bq_client(service_account_path: Path, project_id: str | None = None) -> bigquery.Client:
    credentials = service_account.Credentials.from_service_account_file(str(service_account_path))
    effective_project_id = project_id or credentials.project_id
    if not effective_project_id:
        raise ValueError("Project ID not found. Pass project_id explicitly or ensure it exists in service account JSON.")
    return bigquery.Client(project=effective_project_id, credentials=credentials)

# Optional override. Keep as None to use project_id from service account JSON.
PROJECT_ID_OVERRIDE = None

bq_client = create_bq_client(SERVICE_ACCOUNT_PATH, PROJECT_ID_OVERRIDE)
print(f"Connected to BigQuery project: {bq_client.project}")

Connected to BigQuery project: cognito-dev-380506


In [4]:
sql_text = SQL_FILE_PATH.read_text(encoding="utf-8").strip()
if not sql_text:
    raise ValueError(f"SQL file is empty: {SQL_FILE_PATH}")

query_job = bq_client.query(sql_text)
query_job.result()

print(f"DDL executed successfully. Job ID: {query_job.job_id}")
print(f"State: {query_job.state}")
print(f"Statement type: {query_job.statement_type}")
print(f"Location: {query_job.location}")
if query_job.created and query_job.ended:
    duration_s = (query_job.ended - query_job.created).total_seconds()
    print(f"Duration (s): {duration_s:.2f}")

DDL executed successfully. Job ID: 0d0c73b6-c4cb-4b52-a361-9ca341d2dff7
State: DONE
Statement type: CREATE_TABLE
Location: asia-south1
Duration (s): 0.40


In [5]:
# DDL validation: check destination table metadata when available
if query_job.ddl_target_table:
    target = query_job.ddl_target_table
    print(f"DDL target table: {target.project}.{target.dataset_id}.{target.table_id}")
    table_ref = f"{target.project}.{target.dataset_id}.{target.table_id}"
    table_obj = bq_client.get_table(table_ref)
    print(f"Table exists. Rows: {table_obj.num_rows}, Columns: {len(table_obj.schema)}")
else:
    print("No ddl_target_table metadata returned. This can happen for some DDL statements.")

DDL target table: cognito-dev-380506.data_mart.clinical_trials_tolerability_data_flex_test_temp
Table exists. Rows: 0, Columns: 22
